**Desafío Data Sintética**

**Introducción**
Este documento define el Desafío Técnico para el rol de Coordinador TDM, combinando una presentación formal del objetivo del área con las especificaciones técnicas del ejercicio.

**Objetivo**
La Unidad de Entornos No Productivos – Chapter QA busca garantizar datos adecuados, reproducibles y trazables para pruebas funcionales y E2E, para esto se solicita generar un dataset sintético de clientes bajo un contrato de datos explícito, validarlo contra reglas de negocio y producir artefactos consumibles con trazabilidad (logs, reportes, perfiles).

In [ ]:
# =====================================
# SETUP DEL ENTORNO
# Instalación de dependencias
# =====================================
!pip install pyspark pyyaml faker -q

print("✅ Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.4 MB/s eta 0:00:00
✅ Librerías instaladas correctamente


In [ ]:
# =====================================
# INICIALIZACIÓN DE SPARK
# =====================================
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("TDM_Challenge") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

4.0.2


Generación de 100–500 registros de clientes sintéticos (Parametrizable)

In [12]:
# =====================================
# PARÁMETROS DEL PROCESO
# =====================================

NUM_REGISTROS = 400   # puedes cambiar entre 100–500
PORC_ERROR = 0.0005    # 0.05% porcentaje de errores
# =====================================
# CÁLCULO DE ERRORES
# =====================================

num_errores = max(1, int(NUM_REGISTROS * PORC_ERROR))

print("🚨 Registros con error a inyectar:", num_errores)
print("📊 Registros:", NUM_REGISTROS)
print("⚠️ Porcentaje de error:", PORC_ERROR)

🚨 Registros con error a inyectar: 1
📊 Registros: 400
⚠️ Porcentaje de error: 0.0005


In [14]:
# Crear el YAML corregido y mejorado
yaml_content = """
dataset: clientes_no_productivo
version: 1.1
description: Dataset sintético bancario con soporte multidentificación (EC) y validación cruzada
owner: Unidad de Entornos No Productivos

catalogos:
  tipo_identificacion:
    - CEDULA
    - RUC
    - PASAPORTE

  estado_cliente:
    - ACTIVO
    - INACTIVO

tables:
  clientes:
    primary_key: customer_id

    columns:
      customer_id:
        type: string
        required: true
        unique: true
        pattern: "^Cus[0-9]{3}$"
        description: Identificador único autosecuencial del cliente

      tipo_identificacion:
        type: string
        required: true
        catalog: tipo_identificacion
        description: Define la regla de validación para el campo identificacion

      identificacion:
        type: string
        required: true
        unique: true
        sensitive: true
        description: Número de identificación sintético según tipo documental
        rules:
          - if: "tipo_identificacion == 'CEDULA'"
            pattern: "^[0-9]{10}$"
            validator: "modulo_10_ecuador"

          - if: "tipo_identificacion == 'RUC'"
            pattern: "^[0-9]{13}$"
            validator: "modulo_10_o_13_ruc"

          - if: "tipo_identificacion == 'PASAPORTE'"
            pattern: "^[A-Z0-9]{3,15}$"
            description: Alfanumérico estándar internacional

      nombre:
        type: string
        required: true
        max_length: 50
        sensitive: true
        description: Nombre sintético del cliente

      apellido:
        type: string
        required: true
        max_length: 50
        sensitive: true
        description: Apellido sintético del cliente

      fecha_nacimiento:
        type: date
        required: true
        min_age: 18
        max_age: 90
        format: "%d-%m-%Y"
        sensitive: true
        description: Fecha de nacimiento válida para clientes entre 18 y 90 años

      email:
        type: string
        required: true
        format: email
        sensitive: true
        description: Correo electrónico sintético del cliente

      telefono:
        type: string
        required: true
        pattern: "^09[0-9]{8}$"
        sensitive: true
        description: Teléfono celular sintético del cliente

      direccion:
        type: string
        required: true
        max_length: 100
        sensitive: true
        description: Dirección física del cliente

      fecha_creacion:
        type: timestamp
        required: true
        format: "%d/%m/%Y %H:%M:%S"
        not_future: true
        description: Fecha y hora de creación del registro

      estado_cliente:
        type: string
        required: true
        catalog: estado_cliente
        description: Estado lógico del cliente
"""

with open("contract.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("✅ YAML corregido y creado correctamente")

✅ YAML corregido y creado correctamente


In [15]:
import yaml

# Leer el archivo YAML
with open("contract.yaml", "r", encoding="utf-8") as f:
    contract = yaml.safe_load(f)

print("✅ YAML leído correctamente")
print("Dataset:", contract["dataset"])
print("Versión:", contract["version"])
print("Tablas:", list(contract["tables"].keys()))
print("Columnas de clientes:", list(contract["tables"]["clientes"]["columns"].keys()))
print("Catálogos:", list(contract["catalogos"].keys()))

✅ YAML leído correctamente
Dataset: clientes_no_productivo
Versión: 1.1
Tablas: ['clientes']
Columnas de clientes: ['customer_id', 'tipo_identificacion', 'identificacion', 'nombre', 'apellido', 'fecha_nacimiento', 'email', 'telefono', 'direccion', 'fecha_creacion', 'estado_cliente']
Catálogos: ['tipo_identificacion', 'estado_cliente']


In [19]:
#Validación del contrato de datos YAML
print("Primary key:", contract["tables"]["clientes"]["primary_key"])
print("\nRegla customer_id:")
print(contract["tables"]["clientes"]["columns"]["customer_id"])

print("\nRegla tipo_identificacion:")
print(contract["tables"]["clientes"]["columns"]["tipo_identificacion"])

print("\nRegla identificacion:")
print(contract["tables"]["clientes"]["columns"]["identificacion"])

print("\nCatálogo tipo_identificacion:")
print(contract["catalogos"]["tipo_identificacion"])

print("\nCatálogo estado_cliente:")
print(contract["catalogos"]["estado_cliente"])

Primary key: customer_id

Regla customer_id:
{'type': 'string', 'required': True, 'unique': True, 'pattern': '^Cus[0-9]{3}$', 'description': 'Identificador único autosecuencial del cliente'}

Regla tipo_identificacion:
{'type': 'string', 'required': True, 'catalog': 'tipo_identificacion', 'description': 'Define la regla de validación para el campo identificacion'}

Regla identificacion:
{'type': 'string', 'required': True, 'unique': True, 'sensitive': True, 'description': 'Número de identificación sintético según tipo documental', 'rules': [{'if': "tipo_identificacion == 'CEDULA'", 'pattern': '^[0-9]{10}$', 'validator': 'modulo_10_ecuador'}, {'if': "tipo_identificacion == 'RUC'", 'pattern': '^[0-9]{13}$', 'validator': 'modulo_10_o_13_ruc'}, {'if': "tipo_identificacion == 'PASAPORTE'", 'pattern': '^[A-Z0-9]{3,15}$', 'description': 'Alfanumérico estándar internacional'}]}

Catálogo tipo_identificacion:
['CEDULA', 'RUC', 'PASAPORTE']

Catálogo estado_cliente:
['ACTIVO', 'INACTIVO']


In [20]:
from google.colab import files
files.download("contract.yaml")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>